### **This notebook contains the Python code to conduct a Poisson fixed-effects model on the wildfire and conflict data.**

Task 1: Add data on provinces to the fire spreadsheet. I choose to use the fire data from the J1-VIIRS C2 satellite from NASA MODIS options. You can download more wildfire datasets from NASA MODIS [here](https://firms.modaps.eosdis.nasa.gov/download/list.php)

In [14]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# Load data
fires = pd.read_csv("/Users/hariaksha/Documents/GitHub/climate-conflict/data/climate-pressure/fire/DL_FIRE_J1VIIRS-C2_718929_Nov2000-Feb2026_buffer0km-csv/fire_archive_J1V-C2_718929.csv")                     
provinces = gpd.read_file("/Users/hariaksha/Documents/GitHub/climate-conflict/data/administrative/batas_prov/batas_prov.shp")           

# Change and inspect column names
print(fires.columns.tolist())
provinces.rename(columns={"NAME_1": "provinsi"}, inplace=True)
print(provinces.columns.tolist())

# Convert fires CSV → GeoDataFrame of points
fires_gdf = gpd.GeoDataFrame(
    fires,
    geometry=gpd.points_from_xy(fires["longitude"], fires["latitude"]),
    crs="EPSG:4326"
)

# Reproject provinces to match (EPSG:4326 = standard lat/lon)
provinces = provinces.to_crs("EPSG:4326")

# Spatial join: attach province attributes to each fire point
fires_with_province = gpd.sjoin(
    fires_gdf,
    provinces[["provinsi", "geometry"]],  # keep only cols you need
    how="left",
    predicate="within"
)

# Drop geometry and index columns added by sjoin
fires_with_province = fires_with_province.drop(columns=["geometry", "index_right"])

# Save result 
fires_with_province.to_csv("fires_with_provinces.csv", index=False)
print("Done! Shape:", fires_with_province.shape)
print(fires_with_province[["latitude", "longitude", "provinsi"]].head())


['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_t31', 'frp', 'daynight', 'type']
['provinsi', 'pulau', 'geometry']
Done! Shape: (1307603, 16)
   latitude  longitude    provinsi
0  -4.01907  136.07845       Papua
1  -3.85214  136.35695       Papua
2  -3.98694  136.09912       Papua
3  -4.01018  136.09189       Papua
4  -7.79190  112.01424  Jawa Timur
